# Prepare a Bell state


Material development: **Gage Plott**.

Run this worked example with the video; pause whenever you need time. Nothing is
graded or submitted. No earlier notebook is needed. Start with the
[quickstart](../../intro/START_HERE.md) if the environment is not ready.

## 1. Check the notebook environment

Click a code cell, press **Shift+Enter**, and wait for its running indicator to
finish. Run from top to bottom. A **kernel** is the running Python session: it
remembers names and values until restarted. Choose **Python (PHYS 3810 Intro)**,
then compare the `Interpreter:` path below with the path printed in your activated
setup terminal. A matching display name alone is not enough.

Run this support cell unchanged. `from ... import ...` loads tools we will use.
The first line tells Jupyter to display figures here. The checks below are
software checks, not questions you must answer. If a check fails, return to the
quickstart's recovery steps.

In [ ]:
%matplotlib inline
import struct
import sys
from importlib.metadata import version

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display
from qiskit import ClassicalRegister, QuantumCircuit
from qiskit.primitives import StatevectorSampler
from qiskit.quantum_info import Statevector
from qiskit.visualization import plot_histogram

assert sys.implementation.name == "cpython" and sys.version_info[:2] == (3, 12)
assert struct.calcsize("P") * 8 == 64, "Select 64-bit Python 3.12."
assert version("qiskit") == "2.6.1", "Select the pinned course environment."
assert version("qiskit-aer") == "0.18.2", "Select the pinned course environment."
print("Python:", sys.version.split()[0])
print("Qiskit:", version("qiskit"), "| Aer:", version("qiskit-aer"))
print("Interpreter:", sys.executable)

## 2. Choose the shot count

`=` stores a value under a name. Here `shots` is a whole number. A line beginning
with `#` is a comment for the reader. `print(...)` displays a result.

A **shot** means preparing the circuit again and measuring once. Keep the defaults
for the first run. The simulator seed makes the example reproducible in this
environment: a complete rerun can repeat the same samples. Exact sequences are
not promised across software versions.

In [ ]:
shots = 128  # Change only this number in the follow-along edit.
seed = 2026  # Keep this fixed for the demonstration.
assert isinstance(shots, int) and not isinstance(shots, bool) and shots > 0
print(f"Requested shots: {shots}; simulator seed: {seed}")

## 3. Prepare the Bell state

`QuantumCircuit(2)` starts two qubits in $|00\rangle$. The **Hadamard (H)** gate
acts on qubit 0; then
`cx(0, 1)` applies a controlled X with **qubit 0 as control** and **qubit 1 as
target**. It flips the target for the part of the state where the control is 1.
Together the gates prepare

$$|\Phi^+\rangle = \frac{|00\rangle + |11\rangle}{\sqrt{2}}.$$

The amplitudes in `[00, 01, 10, 11]` order are `[1, 0, 0, 1] / sqrt(2)`.
The plus sign is part of the state, even though the usual measurement histogram
cannot reveal that relative sign. The code checks the ideal state and normalization.

**Bit order:** the diagram usually puts qubit 0 on the top wire. We measure qubit
0 into classical bit 0 and qubit 1 into classical bit 1. The displayed string is
`q1 q0`: **qubit 0 is the rightmost bit**. Thus `01` means q0=1 and q1=0.

This cell starts a new circuit each time. Read the first three lines as the
preparation; the remaining lines are calculation and drawing support.

In [ ]:
bell = QuantumCircuit(2)
bell.h(0)
bell.cx(0, 1)

bell_state = Statevector.from_instruction(bell)
bell_probabilities = bell_state.probabilities()
np.testing.assert_allclose(bell_state.data, np.array([1, 0, 0, 1]) / np.sqrt(2), rtol=0, atol=1e-12)
np.testing.assert_allclose(np.vdot(bell_state.data, bell_state.data), 1, rtol=0, atol=1e-12)
print("Exact probabilities for [00, 01, 10, 11]:", bell_probabilities)
bell_diagram = bell.draw("mpl", scale=1.2)
display(bell_diagram)
plt.close(bell_diagram)

## 4. Sample the pair

A shot prepares the pair again and measures both qubits once. We sample a
measured **copy** with the explicit matching measurement map; `bell` remains
unmeasured. The named classical register `meas` stores the two measured bits.
This ordinary 0/1 measurement is called the **Z basis**.

For this ideal Bell state, the possible results are `00` and `11`, each with
probability one half. Knowing one result predicts the other in this basis.
Each individual bit is still equally likely to be 0 or 1: these are **not two
independent random bits**. Finite counts need not be perfectly balanced.

A classical process choosing `00` or `11` could produce the same histogram.
**This Z-basis histogram alone does not establish entanglement.** The gate
sequence and ideal quantum model tell us which state the circuit prepares.
This is a local classical simulation, not a quantum-device or entropy experiment.

In [ ]:
bell_measured = bell.copy()
bell_measured.add_register(ClassicalRegister(2, "meas"))
bell_measured.measure([0, 1], [0, 1])
bell_result = StatevectorSampler(seed=seed).run([bell_measured], shots=shots).result()[0]
bell_counts = bell_result.data.meas.get_counts()
assert sum(bell_counts.values()) == shots
assert set(bell_counts) <= {"00", "01", "10", "11"}
assert bell.num_clbits == 0
bell_caption = f"Bell Z basis: {shots} simulated shots; seed={seed}; bit order q1 q0."
print(bell_caption)
print("Counts:", bell_counts)

## 5. Plot the counts

Each bar counts pairs with that label. The count total is the number of shots,
not twice the number of shots. The exact probabilities describe repeated ideal
preparations; the bars are a finite sample.

In [ ]:
assert sum(bell_counts.values()) == shots, "Run section 4 after changing shots."
bell_title = f"Bell state, Z basis | {shots} simulated shots | seed {seed}"
bell_histogram = plot_histogram(bell_counts, title=bell_title)
display(bell_histogram)
plt.close(bell_histogram)
print(bell_caption)

## 6. Change one number, then run it

1. In **section 2**, change `shots = 128` to `shots = 1024`.
2. **Run the edited section 2 parameter cell with Shift+Enter.**
3. Run the **section 4 sampling cell**, then the **section 5 plotting cell**.
   Verify the requested total, caption, and plot title now say 1024.
4. If section 7 was already displayed, **rerun its optional X-basis cell too**
   so its count total and title use the current shot count.
5. Change section 2 back to `shots = 128`, **run section 2**, then **sections 4
   and 5** again (and **section 7** if already displayed). Keep seed 2026.

Section 3 is independent of shots. Editing the parameter text without running
its cell leaves the old value in memory; downstream cells can still pass checks
about that old value. For recovery, save your working copy and use
**Kernel > Restart Kernel and Run All Cells**. This notebook works independently
from the RNG notebook.

## 7. Optional: change the measurement basis

This prewritten extension can be skipped in the video. No new parameter edit is
needed. H on **both** qubits just before measurement makes an X-basis measurement;
it starts from a fresh copy of the unmeasured `bell` circuit each time.

In the ideal model, this Bell state still gives matching outcomes. By contrast,
a classical 50/50 mixture of `00` and `11` gives each X-basis outcome probability
1/4. These are different preparations that looked the same in the Z histogram.
Computed probabilities are rounded for display so tiny numerical roundoff
looks like zero. The classical-mixture numbers are exact theoretical values,
not a second set of sampled counts. This short comparison is not a Bell-inequality experiment.

In [ ]:
bell_x = bell.copy()
bell_x.h(0)
bell_x.h(1)
bell_x_probabilities = Statevector.from_instruction(bell_x).probabilities()
classical_mixture_x_probabilities = np.array([0.25, 0.25, 0.25, 0.25])
np.testing.assert_allclose(bell_x_probabilities, [0.5, 0, 0, 0.5], rtol=0, atol=1e-12)

bell_x_measured = bell_x.copy()
bell_x_measured.add_register(ClassicalRegister(2, "meas"))
bell_x_measured.measure([0, 1], [0, 1])
bell_x_result = StatevectorSampler(seed=seed).run([bell_x_measured], shots=shots).result()[0]
bell_x_counts = bell_x_result.data.meas.get_counts()
assert sum(bell_x_counts.values()) == shots
assert set(bell_x_counts) <= {"00", "11"}
assert bell.num_clbits == bell_x.num_clbits == 0
bell_x_caption = f"Bell X basis: {shots} simulated shots; seed={seed}; bit order q1 q0."
print(bell_x_caption)
print("Ideal Bell X probabilities [00, 01, 10, 11] (rounded):", np.round(bell_x_probabilities, 12))
print("Exact classical-mixture X probabilities:", classical_mixture_x_probabilities)
print("Bell X counts:", bell_x_counts)
bell_x_title = f"Bell state, X basis | {shots} simulated shots | seed {seed}"
bell_x_histogram = plot_histogram(bell_x_counts, title=bell_x_title)
display(bell_x_histogram)
plt.close(bell_x_histogram)

## 8. Finish, or explore later

You have prepared a one-qubit random-bit circuit and a two-qubit Bell state.
Both examples use an ideal local simulator. There is nothing to submit.
The [optional notebook menu](../../intro/START_HERE.md#5-optional-exploration)
points to the bounded Grover, QFT, QPE, HHL, and compiled N=15 Shor examples.

Optional references: IBM's [bit-ordering guide](https://quantum.cloud.ibm.com/docs/en/qiskit/bit-ordering)
and [StatevectorSampler documentation](https://quantum.cloud.ibm.com/docs/en/api/qiskit/qiskit.primitives.StatevectorSampler).